In [ ]:
from datetime import datetime
run_start = datetime.now()
print(f"Run start at {run_start.strftime('%Y/%m/%d %H:%M:%S')}")

from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(session)

from snowflake.snowpark.secrets import get_username_password

import requests

#######################################
# NOT USED YET?!?

#import json
#import hashlib
#import re
#from decimal import Decimal
#from typing import Any, Optional
#from snowflake.snowpark import Row
#from snowflake.snowpark.context import get_active_session
#from snowflake.snowpark.functions import col
#from snowflake.snowpark.types import StructType, StructField, StringType, BooleanType


In [ ]:
# Define API endpoints and run identifiers.
api_url = "https://aepcc-api-dev-e0e8bdef741c.herokuapp.com/"
token_url = api_url + "auth/token"
submit_url = api_url + "api/v1/batch/submit"
delete_url = api_url + "api/v1/batch/delete"
tenant_number = "aepcc1yq4xr18ww"
batch_number = run_start.strftime('%Y%m%d_%H%M%S')

# API Credentials
try:
    api_credentials = get_username_password("snowpublic/notebooks/my_password_secret")
    api_username = api_credentials.username
    api_password = api_credentials.password
except:
    print(f"ERROR: API Credentials Not Found. Ensure secrets library was imported and secret is setup. Continuing with hard-coded credentials.")
    api_username = ''   # IMPORTANT: Don't hardcode these in any real projects!
    api_password = ''   # Use a Snowflake Secret and External Access Integration.
    
# Configure ignore and delete rules.
# Ignore will simply trim the client from the payload prior to submission.
# Delete will actively submit a delete request for the client at the end.
ignore_by_metadata = True
delete_by_metadata = True
delete_by_key_not_found = True

# Truncate log tables?
truncate_run_history = False
truncate_client_snapshot = False
truncate_field_errors = True

# Developer Assists
skip_api_calls = True


In [ ]:
# Set default database and schema.
source_database = 'DB_DANIEL'
source_schema = source_database + '.SHARE_TO_API'

# Define entire structure of final payload.
payload_metadata = {
    'tenantNumber': {
        'type': 'literal',
        'value': tenant_number
    },
    'batchNumber': {
        'type': 'literal',
        'value': batch_number
    },
    'clientRecords': {
        'type': 'base_table',
        'location': source_schema + '.CLIENTS',
        'values': {
            'clientNumber': {
                'type': 'field',
                'name': 'clientId'
            },
            'description': {
                'type': 'field',
                'name': 'description'
            },
            'ignoreClient': {
                'type': 'embedded_field',
                'name': 'ignoreExample',
                'within': 'consent',
                'TODO IGNORE_ON': ''
            },
            'deleteClient': {
                'type': 'embedded_field',
                'name': 'deleteExample',
                'within': 'consent',
                'TODO DELETE_ON': ""
            },
            'reviews': {
                'type': 'child_table',
                'location': source_schema + '.REVIEWS',
                'match_on': {'parent': 'clientId', 'child': 'clientId'},
                'values': {
                    'reviewNumber': {
                        'type': 'field',
                        'name': 'reviewId'
                    },
                    'date': {
                        'type': 'field',
                        'name': 'date'
                    },
                    'description': {
                        'type': 'field',
                        'name': 'description'
                    },
                    'review_items': {
                        'type': 'child_table',
                        'location': source_schema + '.REVIEW_ITEMS',
                        'match_on': {'parent': 'reviewId', 'child': 'reviewId'},
                        'values': {
                            'description': {
                                'type': 'field',
                                'name': 'description'
                            }
                        }
                    }
                }
            },
            'hospitalisations': {
                'type': 'child_table',
                'location': source_schema + '.HOSPITALISATIONS',
                'match_on': {'parent': 'clientId', 'child': 'clientId'},
                'values': {
                    'reviewNumber': {
                        'type': 'field',
                        'name': 'hospitalisationId'
                    },
                    'description': {
                        'type': 'field',
                        'name': 'description'
                    }
                }
            }
        }
    }
}


In [ ]:
create or replace TABLE HOSPITALISATIONS(
	"clientId" INTEGER,
	"hospitalisationId" TEXT,
	"description" TEXT
);

create or replace TABLE REVIEW_ITEMS(
    "reviewId" TEXT,
    "description" TEXT
);

In [ ]:
if skip_api_calls:
    print(f"skip_api_calls is set to TRUE")
    
else:
    # Request an access token from the API.
    token_resp = requests.post(
        token_url,
        json={
            "client_id": api_username,
            "client_secret": api_password,
        },
        headers={"Content-Type": "application/json"},
        timeout=30,
    )
    
    # Extract the token from the API response.
    api_token = token_resp.json()["access_token"]
    
    # And a partially-masked version for run history logs.
    log_api_token = api_token[:3] + '...' + api_token[-10:]
    
    # Log API response details and cell execution status.
    print(f"API Response Status: {str(token_resp.status_code)}")
    print(f"API Response Text:\n{token_resp.text}")
